# Bulk modulus — single volumetric-strain level

Analysis of **one** level of a `compress_slab` run: drained **isotropic** compression of an isolated gel at
**constant bath pressure** — two wet NPT-pistons hold the bath at `P_target` and six solvent-transparent plates,
seated onto the faces of the gel, close together to a volumetric strain `ε_vol`.  The deck and this notebook are the
bulk-modulus counterparts of `triaxial_compression_two_pist` and `triaxial_compression_single_two_pist.ipynb`: the
mechanism that gives `M` there gives `K` here.  Set `LEVEL` and the run identifiers in **Config**, run the
**sync** cell once, then run everything.  All analysis code lives in `scripts/lib/bulk.py` (on top of
`scripts/lib/triaxial.py`); this notebook only configures it and draws the figures.

Method notes and the file list are in **Notes** at the end (2026-09-29).

## 1 · Setup and computation

In [ ]:
import sys, importlib
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

LIB = Path('lib').resolve()            # scripts/lib: bulk.py (this analysis) on top of triaxial.py (shared machinery)
if str(LIB) not in sys.path:
    sys.path.insert(0, str(LIB))
import triaxial as tri
import bulk
tri = importlib.reload(tri)            # pick up edits to lib/*.py without a kernel restart
bulk = importlib.reload(bulk)
bulk.setup_style()
print('analysis code: ', LIB / 'bulk.py')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  CONFIG -- the only cell to edit when switching runs
# ══════════════════════════════════════════════════════════════════════════
LEVEL = "0.10"      # the ONE volumetric-strain level analysed here (one of STRAIN_TARGETS in the .batch; files _c<LEVEL>)
cfg = bulk.Config(
    DATANAME    = "final_config_slab_support_5beads_square_rho10_nowalls_1.0_1.0_8000000_six_plates",
    INTERACTION = "1.0_1.0",          # epsSS_epsSP
    NSTEPS      = None,           # <steps> tag = each level's auto-sized HOLD LENGTH; None resolves it from the files
    RUN_ID      = "bulk_compression_square_rho10_010_unload_01",  # local folder under flow_data_local/{bulk,plots/bulk}: the compress_slab_single.batch
                                  #  run (one level 0.10 + the unload, 2026-10-28).  sweep_04 (no unload) works too with LEVEL "0.11":
                                  #  the D_m parts then print 'no unload files' and are skipped
    COMP_LEVELS = [LEVEL],
    AXES        = ('z', 'x', 'y'),  # profile axes; the first is the one the profile figures draw by default
    # ---- measurement windows ----
    plateau_frac      = 0.25,
    plateau_frac_auto = 0.45,
    P_BARO = 1.5,                 # = PRESS_TARGET of the batch: normalises the total-stress panels, line of the bath check
    # ---- K as an increment from the eps_vol = 0 reference ----
    K_SUBTRACT_REF = True,
    K_STRAIN   = 'disp',          # eps_vol in the denominator: 'disp' (network strain from the steady cumulative displacement
                                  #  profiles along x, y, z -- decks since 2026-10-05; falls back rg -> plate) | 'plate' (applied) |
                                  #  'rg' | 'bb' (the network's own) | 'nominal'
    PLATE_AREA = 'gap',           # plate pressure = load / face enclosed by the other plates ('bb': live bounding-box face)
    # ---- solvent volume fractions in the core columns (lib/volfrac.py) ----
    VOR_ENABLE = True, REF_VOR_FRAMES = 3, VOR_MAX_FRAMES = 4, VOR_EVO_FRAMES = 5, P_CAL = 1.5, P_CAL_MODE = 'const',
    # ---- closure with the triaxial / shear notebooks (None = skip) ----
    M_REF = None,                 # M of the SAME gel from triaxial_compression_*_two_pist  ->  G = 3/4 (M - K), confined D_c = (M/K) D_m
    G_REF = None,                 # G to draw next to it
    # ---- the unload (deck Phase 2c, UNLOAD=1 in the batch): free re-swelling -> D_m = (K/M) D_c, kappa = D_m/K ----
    DM_UNLOAD = True, UNLOAD_L = 'hold', DM_N_MODES = 3, DM_TRACE_SKIP = 0.0, DM_N_BOOT = 200,
)
# Every other knob keeps its default -- see `bulk.Config` in lib/bulk.py and `tri.Config` in lib/triaxial.py.
# stress_aniso file of the slab_with_support run that produced the input (free swelling of the isolated gel), for the
# whole-run anisotropy figure; None = the compress_slab run alone
SWELL_ANISO = None

In [ ]:
# Pull the files this notebook reads from Expanse in ONE login (password + TOTP prompts).
# Runs made on Pod or Bridges-2: set HOST='pod' / 'bridges' in the Config, leave SYNC on (it then only prints
# where to put the files) and copy the run's output_files/*/*.dat into flow_data_local/bulk/<RUN_ID>/ by hand.
SYNC, FORCE_SYNC = True, False
if SYNC:
    bulk.sync_from_expanse(cfg, levels=[LEVEL], force=FORCE_SYNC)

In [ ]:
# Load + compute everything:
#   R  -- the eps_vol = 0 reference state: seated plates, reference volumes, plate preload, per-axis reference profiles
#   L  -- this level: measured strains, per-axis stresses + Terzaghi split, plate plateau, K, isotropy, bath check; with an unload
#         (files _u<LEVEL>, deck Phase 2c) also L['U'] (the re-swelling) and L['Dm'] (its D_m fits) -> kappa = D_m/K
#   add_volume_fractions -- Voronoi + calibrated phi_s in the core columns (trajectories; cached in vor_cache/)
R = bulk.load_reference(cfg)
L = bulk.load_level(cfg, R, LEVEL)
assert L is not None, f'level {LEVEL}: core files missing -- run the sync cell'
bulk.add_volume_fractions(cfg, R, [L])
bulk.print_summary(cfg, [L])

## 2 · Figures

In [ ]:
# 1 · Strain diagnostic -- bold: applied (plate) ε_vol; solid: network 1 − V_Rg/V_Rg,0; dashed: network V_BB; shaded = plateau window;
#     with an unload the curves continue through the retraction (dash-dot = plates open) and the free re-swelling
bulk.fig_strain(cfg, R, [L]);

In [ ]:
# 2 · Solvent volume fraction φ_s (mass fraction, Voronoi, λ-calibrated) in the core column: reference vs compressed,
#     one figure per axis
bulk.fig_volfrac(cfg, R, L, axes=cfg.AXES);

In [ ]:
# 2b · φ_s evolution over the hold along x, y, z (λ-calibrated Voronoi; key='phi_vor' or 'phi_mf' for the other estimators)
bulk.fig_volfrac_evolution(cfg, R, L, key='phi_cal');

In [ ]:
# 3 · Total stress evolution σ^t_zz, σ^t_xx, σ^t_yy / P_bath -- reference (dashed) -> hold (cividis) -> plateau (bold).
#     Profile along cfg.AXES[0]; pass axes=cfg.AXES for all three columns
bulk.fig_total_stress(cfg, R, L);

In [ ]:
# 4 · Solvent and polymer partial σ_zz evolutions with the total superimposed
bulk.fig_partial_stress(cfg, R, L);

In [ ]:
# 5 · Network stress σ'_zz, σ'_xx, σ'_yy (Terzaghi: σ' = σ^t − p_pore): reference (ε_vol = 0, dashed) and FINAL
#     equilibrated state only, 95 % bands -- drawn along all three axes (one figure per axis)
bulk.fig_network_stress(cfg, R, L, axes=cfg.AXES);

In [ ]:
# 6 · Plate pressures P = load / face area vs step: the six plates and their mean, linear + log; green = plateau window
bulk.fig_plates(cfg, R, L);

In [ ]:
# 7 · Bulk modulus K (increments from ε_vol = 0): network (per profile axis and their mean) vs plates
bulk.fig_K(cfg, R, L);

In [ ]:
# 8 · Isotropy of the response: Δσ'_xx, Δσ'_yy, Δσ'_zz per profile axis and the six plate pressure increments
#     (equal bars = no deviator; replaces the anisotropy-ratio and G figures of the triaxial notebooks)
bulk.fig_isotropy(cfg, R, L);

In [ ]:
# 8b · Network-stress anisotropy over the hold: σ'_ii/Π per profile axis (absolute) and Δσ'_ii/ΔΠ (increments)
bulk.fig_ratio(cfg, R, L);

In [ ]:
# 8c · Whole-run polymer partial-stress anisotropy σ_p,xx/σ_p,zz, σ_p,yy/σ_p,zz (stress_aniso, from step 0; the hold shaded)
#      -- next to the free swelling of the slab_with_support run when SWELL_ANISO points at its stress_aniso file
bulk.fig_anisotropy_run(cfg, R, [L], swell_file=SWELL_ANISO);

In [ ]:
# 9 · The free re-swelling after the hold (unload, deck Phase 2c) and its diffusivity D_m = (K/M) D_c: (a) the Rg edges growing
#     back with the fitted free-cube series, (b) every estimator (edges, Rg volume strain, interior Π) collapsed on 1 − R(t),
#     (c) D_m, τ₁ = L²/(3π² D_m), κ = D_m/K; bottom: the cumulative-displacement lag per axis with the cos(kπζ) model.
#     Skipped (with a note) when the run has no unload files.
bulk.fig_Dm(cfg, R, L);

In [ ]:
# 10 · Closure M = K + 4G/3 -- needs M_REF in the Config (M of the same gel, triaxial notebook); with an unload also κ = D_m/K
#      and the confined D_c = (M/K) D_m a triaxial hold of this gel would show
bulk.fig_closure(cfg, R, L);

In [ ]:
# 11 · Bath check -- P_feed and P_perm measured on the wet pistons vs P_target over the hold
bulk.fig_wet_pistons(cfg, R, L);

In [ ]:
# 13 · Thermodynamic pressure P_th = −⅓ tr(σ^t) evolution (positive under compression; dotted = P_bath)
bulk.fig_thermo_pressure(cfg, R, L);

In [ ]:
# 14 · Osmotic pressure Π = −⅓ tr(σ′): reference (ε_vol = 0) and final equilibrated state, 95 % bands, along all
#      three axes -- the quantity K_network is read from
bulk.fig_osmotic_pressure(cfg, R, L, axes=cfg.AXES);

## Notes

Everything below is reference material — nothing above depends on reading it.

### Geometry and files (2026-09-29)

* **Geometry**: an isolated gel block in its bath; `x`, `y` periodic, `z` closed by two wet NPT-pistons (feed above,
  permeate below) that hold the bath at `P_target`.  Six solvent-transparent plates load the six faces of the gel.
  Atom types: 4 = plate z-lo, 7 = plate z-hi (the *support* and the *load piston* of the two-piston decks),
  8/9 = plates x-lo/x-hi, 10/11 = plates y-lo/y-hi, 5/6 = feed / permeate piston.
* **Core columns**: every profile is taken in a column through the gel — along `z` the cross-section is
  `[plate x-lo + core_margin, plate x-hi − core_margin] × [same in y]`, likewise along `x` and `y` — so each profile
  reads *bath | gel | bath*.  The deck divides the per-atom stress by the **live** column cross-section, so the
  files hold true stresses at every strain (the 2026-08 deck divided a static atom group by a fixed volume).
* **Files**: along `z` the profile files carry the two-piston names (`sigmazz_polymer_…`); along `x`, `y` they are
  tagged `_alongx`, `_alongy`.  `plate_position`, `plate_force[_avg]` hold one column per plate
  (`zlo zhi xlo xhi ylo yhi`; loads are compressive, `N > 0` when the gel pushes the plate outward);
  `strain_vol` holds the plate and network volumetric strains.  Every level is tagged `_<hold>_c<level>`, the
  reference window `_ref`.
* **Profile figures along another axis**: every profile figure takes `axes=`, e.g.
  `bulk.fig_network_stress(cfg, R, L, axes=('x', 'y', 'z'))`; the files are then tagged `_alongx` / `_alongy`.

### How K is measured

* **Terzaghi**: `σ′ = σᵗ − p_pore`, with `p_pore` the total stress in the **bath bins of the same column**, on
  both sides of the gel (≥ `res_gel_gap_bins` bins outside the plates; along `z` also ≥ `res_wall_margin` = 3 σ from
  the wet pistons, whose sheets deplete the solvent and keep half of their pair virial).
* **Osmotic pressure** `Π = −⅓ tr σ′` (positive under compression).  The profiles store compression-positive
  stresses, as every triaxial profile, so in the files it is the mean of the three `σ′_ii`.
* **K_network** `= (⟨Π⟩_int,plateau − Π_ref)/ε_vol`: plateau-averaged profile (last `plateau_frac` of the hold),
  mean over the wall-trimmed interior bins, one estimate per profile axis.  The headline value is the mean of the
  axes; its CI is the mean of their half-widths (the columns overlap at the gel centre, so they are not independent).
* **K_plates** `= (⟨P̄⟩_plateau − P̄_ref)/ε_vol`, `P̄` = mean over the six plates of load / face area, read over the
  longest drift-free trailing window (circular block bootstrap) — the counterpart of `M_piston`.
  `PLATE_AREA='gap'` takes the face enclosed by the other two plate pairs; `'bb'` the live bounding-box face.
* **Increments**: both estimators subtract **their own** `ε_vol = 0` reading (`K_SUBTRACT_REF`), measured in the
  reference window (Phase 1.5), exactly as `M_SUBTRACT_REF`.
* **Strain**: `ε_vol` is the *measured* plate strain `1 − Π(1 − closure_i/L0_i)`; `L0_i` = seated plate gap −
  2 `contact_gap`.  The network's own `V_Rg` and `V_BB` strains are printed next to it.  The plates reach the
  network through its sparse outer shell, so the network strain can fall short of the plate strain;
  `K_net(Rg)` is the same stress increment over the network's own Rg strain; with `K_STRAIN='disp'` (default) the headline K already divides by the network strain from the steady displacement-profile slopes (`disp_*_polymer_cum`, decks since 2026-10-05), and `'rg'` or `'plate'` select the other denominators.
* **Seating** (`SEAT_MODE` in the batch): `rg` seats the plates on the Rg-based faces of the gel, `bb` on the
  outermost beads.  The bounding box of the 12-cell cube is 3–5 σ per side wider than its Rg edge while a 10 %
  level closes each face by ~1.2 σ, so `bb` seating spends the level on the shell.  With `rg` the folded shell
  carries a preload, which is the `P̄_ref` every K subtracts.
* **Unrelaxed hold** (`RELAX_SYS`): an exponential tail fitted to the mean plate pressure; the plateau's excess
  over the fitted asymptote, / ε_vol, is `δ_sys`, added to the **lower** bound of both K.
* **Isotropy**: see *Network anisotropy* below.  Under isotropic compression the deviator of `Δσ′` vanishes by symmetry — there is no `G` in this
  measurement, which is why the anisotropy-ratio and `G` figures of the triaxial notebooks have no counterpart
  here.  A deviator in the isotropy figure is a loading defect (unequal plates, a gel that turned).
* **No diffusivity from the hold** (2026-10-28): the diffusivity is `D_m` of the free re-swelling after the hold —
  see *D_m: the free re-swelling after the hold* below.
* **Closure**: with `M_REF` = the `M` of the same gel from the triaxial notebooks, `G = ¾(M − K)` and the confined
  `D_c = (M/K) D_m`; `G_REF` (shear or triaxial notebook) is drawn next to it.  `κ = D_m/K` needs no `M_REF`.

### What replaced the 2026-08 notebook

`archive/bulk_modulus_analysis.ipynb` derived an osmotic `K(t) = −V dΠ/dV` from the whole-run Rg path (its
"active" mask also caught the NPH equilibration), and a drained `K` from a fit over stages recovered by
step-gap clustering with a fixed count of 3, without the reference window the deck recorded.  Here every level is
its own file set, the reference window is the zero of every modulus, and no stage count is assumed anywhere.
### Network anisotropy

* **Over the hold** (`fig_ratio`): `σ′_ii/Π` in the gel interior along every profile axis (1 = isotropic network
  stress) and the increments `Δσ′_ii/ΔΠ` from the ε_vol = 0 reference — the response to the plates alone.
* **Over the whole run** (`fig_anisotropy_run`): `σ_p,xx/σ_p,zz`, `σ_p,yy/σ_p,zz` from the `stress_aniso` file the
  deck writes from step 0 (settle, seating, reference, every level; the holds are shaded).  `slab_with_support`
  writes the same file, so `SWELL_ANISO` = its `stress_aniso_<run>.dat` draws the free swelling of the isolated gel in
  a first panel — for an isolated gel this file and the Rg edges `L_x, L_y, L_z` are what measure the network's
  anisotropy; the box faces are not the gel's faces (see the barostat note in `slab_with_support.lmp`).

### Volume fractions in the core columns

`add_volume_fractions` tessellates `traj_ref` (`REF_VOR_FRAMES` frames) and `traj_stress` of every level
(`VOR_MAX_FRAMES` frames in the plateau + `VOR_EVO_FRAMES` frames over the whole hold) once per frame — the
periodic voro++ pass of `lib/volfrac.py` — and bins the solvent cell volumes inside the core column along `x`, `y`
and `z` (`core_margin` inside the plates, as the deck's stress columns).  Three estimators, as in the triaxial
notebooks: mass fraction `ρ_s/ρ_bath` from the density profiles, Voronoi `φ_s^vor` (column bin volume), and the
λ-calibrated `φ_s^cal = λ(φ_p, P)·φ_s^vor` (exact-saturation norm; `P` per bin from `P_CAL_MODE`: `const` = `P_CAL`,
`pore` = the axis' bath baseline, `thermo` = the local `P_th`).  `fig_volfrac` shows reference vs plateau per axis,
`fig_volfrac_evolution` the evolution over the hold along the three axes.  Frames are cached in
`flow_data_local/bulk/<RUN_ID>/vor_cache/`.

### D_m: the free re-swelling after the hold (2026-10-28)

* **Why the hold is not used.** The hold-onset `displace/atom` files were a one-snapshot reference that the block
  averages carried as a static pattern (the fits pinned `D_c` at the search bound), and a compression hold on a
  `bb`-seated cube mixes the folding of the loose shell in with the drainage (two time scales, see the triaxial
  notes).  The deck no longer writes those files and the notebooks take **no diffusivity from the compression**;
  the compression gives `K` only.
* **The unload** (deck Phase 2c, `UNLOAD=1` in both batches): after the hold the six plates open at the drive speed
  to the seated gaps + 2 `UNLOAD_MARGIN` (3 σ per plate, clear of the re-swollen shell) and freeze; the gel re-swells
  **freely** for `UNLOAD_STEPS` (0 = the hold length), files `_<hold>_u<level>`, the level's full file set.
* **`D_m` instead of `D_c`.** In the triaxial holds the solvent drains along one axis under lateral constraint and
  the dilatation diffuses with the *confined* coefficient `D_c = M/ζ`.  The cube released on all six faces is the
  three-dimensional *unconstrained* problem: its volumetric response is set by the bulk modulus, so the diffusivity
  it measures is `D_m = K/ζ = (K/M) D_c`.  (The Tanaka–Fillmore bulk equation keeps `M` in the diffusion term and
  lets `K` enter through the stress-free boundary; the `D_m` convention folds that into the coefficient.  Either way
  the number to compare with the triaxial unloads is `(K/M) D_c`.)
* **The model** (`bulk.fit_Dm_unload`): the faces are unloaded within a few τ and the dilatation excess starts
  uniform (the compressed state): `∂e/∂t = D_m ∇²e`, `e = e_∞` on the six faces — the separable product of three slab
  solutions.  The fraction of the initial excess still inside the cube,
  `R(t) = M(t; L_x) M(t; L_y) M(t; L_z)`, `M(t; L) = Σ_odd 8/(kπ)² exp(−k²π² D_m t/L²)`,
  is what every observable linear in the dilatation follows: each Rg edge and the Rg volume strain grow as `1 − R(t)`,
  the interior `Π` excess decays as `R(t)`, and the cumulative displacement along an axis, referenced to its re-swollen
  plateau, relaxes in odd `cos(kπζ)` modes (u = 0 at the centre) at `exp(−k²π² D_m t/L_a²)` times the transverse
  `M_b M_c`.  The fundamental `(1,1,1)` decays at `τ₁ = L²/(3π² D_m)`, three times faster than a slab of the same
  thickness.  Lengths = the compressed Rg edges at the hold end (`UNLOAD_L = 'hold'`; `'ref'` = the edges the level
  drove from).  Time runs from the start of the retraction; the fits start when the plates have stopped.
* **Estimators** (figure 9): the three **Rg edges** jointly (headline; free amplitude per axis whose ratio to
  `L_start − L_hold` is the *recovery* of the compression; 95 % interval from a circular block bootstrap of the
  residuals), the **Rg volume strain**, the **interior `Π`** block means of the three profile axes, and the
  **displacement profiles** per axis and jointly (`DM_N_MODES` odd cosines, a free constant per block for the
  floating gel's COM drift).  The BB edges are drawn faint only: the outermost beads spring back with the plates.
  A fit at a `DM_BOUNDS` edge is flagged, not reported.
* **`κ = D_m / K`** from the same run (`K_network` of the hold); with `M_REF` also the confined `D_c = (M/K) D_m`
  a triaxial hold of this gel would show, and `G = ¾(M − K)`.
